# Optical-Pump Time Scan Analysis

Interactive analysis of `OpticalPumpScan` runs (mF=0 pumping fidelity vs. optical-pump TIME).
Reuses the BLAS-free helpers from `analyze_optical_pump_scan.py`.

For each pump time, two interleaved configs (pi / no-pi reference). Among loaded atoms
`lost = 1 - retention` (atom ended in F=4, blown away). The clean mF=0 transfer signal
subtracts the F=4-leftover background: **`signal(t) = lost_pi(t) - lost_noPi(t)`**, which
rises and **saturates**. We fit `signal(t) = S_inf - (S_inf - S0) exp(-t/tau)` and report
`t_rise98 = tau*ln(50)` as the **recommended `t_optical_pump`** (recommend-only; not written).

Run with the **working-BLAS `python`** kernel (numpy / h5py / mpl); matplotlib crashes in
the artiq7_claude env.

In [ ]:
import sys, os, glob
from pathlib import Path
_ANALYSIS_DIR = r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\analysis"
if _ANALYSIS_DIR not in sys.path:
    sys.path.insert(0, _ANALYSIS_DIR)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2
from analyze_optical_pump_scan import (
    RESULTS_DIR, find_run, load_run, retention_table, build_curves,
    fit_saturation, empirical_knee, wilson,
)
print("Results dir:", RESULTS_DIR)

## 1. List recent runs

In [ ]:
files = sorted(glob.glob(str(RESULTS_DIR / "**" / "*-OpticalPumpScan.h5"), recursive=True),
               key=lambda f: int(os.path.basename(f).split("-")[0]))
if not files:
    print("No *-OpticalPumpScan.h5 found yet.")
for f in files[-12:]:
    rid = int(os.path.basename(f).split("-")[0])
    d, s, _ = load_run(f)
    n = len(np.atleast_1d(d.get("op_t_pump_us", [])))
    nl = int(np.sum(np.atleast_1d(d.get("op_atom_loaded", [0])).astype(int)))
    print(f"RID {rid:>5} | {n:>6} pts | {nl:>5} loaded | "
          f"live t_optical_pump={float(s.get('t_optical_pump',0))*1e6:.1f} us | "
          f"t_pump=[{s.get('t_pump_min_us','?')}..{s.get('t_pump_max_us','?')}] us")

## 2. Select a run

Set `RID` to an int, or `None` for the latest.

In [ ]:
RID = None   # <-- int or None (latest)

path = find_run(RID)
data, snap, rid = load_run(path)
curves = build_curves(data)
ts = curves["ts"]
fit = fit_saturation(ts, curves["signal"], weights=curves["n_eff"])
knee = empirical_knee(curves)

print("File :", os.path.basename(path))
print(f"RID {rid} | {len(ts)} pump-time points | live t_optical_pump = "
      f"{float(snap.get('t_optical_pump',0))*1e6:.1f} us")
print("  t_pump(us)   lost_pi   lost_noPi   signal   n_pi  n_noPi")
for i in range(len(ts)):
    print(f"  {ts[i]:9.2f}   {curves['lost_pi'][i]:7.3f}   {curves['lost_no'][i]:9.3f}   "
          f"{curves['signal'][i]:6.3f}   {int(curves['n_pi'][i]):4d}  {int(curves['n_no'][i]):4d}")
if fit.get("ok"):
    print(f"\nFIT: S_inf={fit['S_inf']:.3f}  S0={fit['S0']:.3f}  tau={fit['tau']:.1f} us  "
          f"-> recommended t_rise98 = {fit['t_rise98']:.1f} us")
if knee is not None:
    print(f"empirical knee: t_pump = {knee:g} us")

## 3. Panel A — lost fraction: pi vs no-pi reference

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.errorbar(ts, curves["lost_pi"],
            yerr=[np.nan_to_num(curves["lost_pi"]-curves["lpi_lo"]),
                  np.nan_to_num(curves["lpi_hi"]-curves["lost_pi"])],
            fmt="o-", color="C0", capsize=3, label="lost_pi (pi applied)")
ax.errorbar(ts, curves["lost_no"],
            yerr=[np.nan_to_num(curves["lost_no"]-curves["lno_lo"]),
                  np.nan_to_num(curves["lno_hi"]-curves["lost_no"])],
            fmt="s-", color="C1", capsize=3, label="lost_noPi (reference)")
ax.set_xlabel("optical-pump time (us)")
ax.set_ylabel("lost fraction  (1 - retention)")
ax.set_ylim(-0.02, 1.02)
ax.set_title(f"Lost fraction — RID {rid}")
ax.legend(loc="best"); ax.grid(alpha=0.3)
plt.show()

## 4. Panel B — clean mF=0 signal + saturating fit → recommended time

In [ ]:
sig_half = np.sqrt(np.nan_to_num((curves["lpi_hi"]-curves["lpi_lo"])/2.0)**2
                   + np.nan_to_num((curves["lno_hi"]-curves["lno_lo"])/2.0)**2)
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.errorbar(ts, curves["signal"], yerr=sig_half, fmt="o", color="C3",
            capsize=3, label="signal = lost_pi - lost_noPi")
t_max = float(np.nanmax(ts)) if len(ts) else 1.0
if fit.get("ok"):
    tt = np.linspace(0.0, t_max, 300)
    yy = fit["S_inf"] - (fit["S_inf"] - fit["S0"]) * np.exp(-tt / fit["tau"])
    ax.plot(tt, yy, "-", color="C3", alpha=0.7, label=f"fit (tau={fit['tau']:.0f} us)")
    ax.axvline(min(fit["t_rise98"], t_max), color="k", ls="--", alpha=0.7,
               label=f"t_rise98 = {fit['t_rise98']:.0f} us")
if knee is not None:
    ax.axvline(knee, color="g", ls=":", alpha=0.7, label=f"empirical knee = {knee:g} us")
ax.set_xlabel("optical-pump time (us)")
ax.set_ylabel("clean mF=0 signal")
ax.set_title("mF=0 transfer signal (background-subtracted)")
ax.legend(loc="best"); ax.grid(alpha=0.3)
plt.show()
if fit.get("ok"):
    print(f"RECOMMENDED t_optical_pump ~ {min(fit['t_rise98'], t_max):.1f} us "
          f"(set it yourself on the dashboard; this notebook does NOT write it).")

## 5. (optional) one-shot PNG via the script

`analyze(path)` writes the 2-panel PNG next to the `.h5`.

In [ ]:
# from analyze_optical_pump_scan import analyze
# analyze(path)